# ⚠ DO NOT RUN unless the group decided to RETRAIN
This notebook **retrains all three models and overwrites them on Hugging Face** (hansjio/tsektxt-*). The thesis models are final (trained 2026-11-01).
To test the models or score the Taglish set, use **notebooks/colab_eval.ipynb** instead. To present results, use **notebooks/colab_present.ipynb**.

---
# TsekTxt – retrain all three models (Colab)
**Reading this to explain fine-tuning?** You can read it without running anything. Each step below says what it does. *Fine-tuning* means taking a pre-trained language model and continuing its training on our labelled news articles so it learns the credible / not credible task. The same fixed configuration (`configs/config.yaml`, no hyperparameter search, one seed) is used for all three models.
**Before running (retraining only):** Runtime → Change runtime type → **T4 GPU** (or better). Add a Colab *Secret* (key icon on the left) named `HF_TOKEN` = a Hugging Face token with **Write** access, and turn on "Notebook access".
The GitHub repo must already contain the NEW code and `data/raw/combined_dataset_clean.csv` (push them first). Run the cells top to bottom.

## 1. Check the GPU
**What it does:** shows which GPU Colab gave you. Fine-tuning a transformer is very slow without one. **Output:** GPU name and `CUDA available: True`.

In [ ]:
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. Get the code (repo is public, so no GitHub token is needed)
**What it does:** downloads the repository fresh (deleting any old copy in this runtime) and installs the libraries from `requirements.txt`. **Input:** GitHub. **Output:** the code folder, ready to use. **Time:** a minute or two.

In [ ]:
import os
os.chdir('/content')
!rm -rf tsektxt-model-training
!git clone https://github.com/hansjio/tsektxt-model-training.git
%cd tsektxt-model-training
!pip install -q -r requirements.txt

## 3. Log in to Hugging Face using the Colab secret
**What it does:** logs in with your write token so trained models can be uploaded later (Section 5b). **Input:** the `HF_TOKEN` secret. **Output:** "Logged in as: ..." which must match the owner in `configs/config.yaml`. Never paste a token into the notebook.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import whoami
print('Logged in as:', whoami()['name'])   # must match hub.user in configs/config.yaml

## 4. Clean the data and make the 70/15/15 split
**What it does:** cleans the article text, then splits it into 70% training (the model learns from it), 15% validation (chooses when to stop) and 15% test (final score, never used for learning). The Taglish posts are not part of this.
**Input:** `data/raw/combined_dataset_clean.csv`; if it is missing, upload it with the Files panel on the left (drag into `data/raw/`). **Output:** `data/processed/` splits. **Time:** a few minutes.

In [ ]:
!ls -la data/raw
!python src/clean_data.py --input data/raw/combined_dataset_clean.csv
!python src/make_splits.py

## 5. Trial run first (one model, NOT uploaded). Check it finishes and prints metrics.
**What it does:** fine-tunes one model (XLM-R) end to end without uploading, to catch mistakes before the long run. It trains with a fixed learning-rate schedule, watches validation loss each epoch and stops early when it stops improving. **Output:** training log and test metrics in `results/`.

In [ ]:
!python src/run_all.py --only xlmr

## 5b. Full run: trains all three, uploads each to Hugging Face, then compares (an hour or so on a T4 — an estimate, not measured)
**What it does:** repeats Section 5 for RoBERTa-Tagalog, mBERT and XLM-R, **uploads each to Hugging Face (overwriting the current models)**, then writes `results/comparison.md`. **Only run this if the group decided to retrain.**

In [ ]:
!python src/run_all.py --push

## 6. (Later) Evaluate on your Taglish election test set
**What it does:** scores the finished models on the Taglish posts (test only, never trained on). **Input:** `taglish_test.csv` (columns `id,text,label`) in `data/external/`. **Output:** `external_*` metrics and an updated `comparison.md`. For the full Taglish workflow prefer `colab_eval.ipynb`.

In [ ]:
for m in ['roberta-tagalog','mbert','xlmr']:
    !python src/evaluate_external.py --model {m} --csv data/external/taglish_test.csv
!python src/compare.py

## 7. Download the results so you can commit them to GitHub
**What it does:** zips `results/` (without checkpoints) and the cleaning report and downloads it, because Colab files disappear when the runtime ends. **Output:** `results.zip`.

In [ ]:
!zip -qr results.zip results data/processed/cleaning_report.json -x 'results/*/checkpoints/*'
from google.colab import files; files.download('results.zip')